problem: https://judge.nitro-ai.org/competitions/nitro/pre-iaio-2026/2/view

In [18]:
import re
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [34]:
def parse_target(target_str):
    match = re.search(r'\((-?\d+),\s*(-?\d+)\)', target_str)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return None

def parse_vectors(vectors_str):
    matches = re.findall(r'\((-?\d+),\s*(-?\d+)\)', vectors_str)
    return [(int(x), int(y)) for x, y in matches]

sys.setrecursionlimit(50000)
class Game:
    def __init__(self, row):
        self.n = int(row['N'])
        self.b = int(row['B'])
        self.target = parse_target(row['Target'])
        self.vectors = parse_vectors(row['Vectors'])

    def manhattan(self, x, y):
        return np.abs(x-self.target[0]) + np.abs(y-self.target[1])

    def best_move(self, x, y, used_mask):
        best_gain = -float('inf')
        best_i = -1

        old_dist = self.manhattan(x, y)

        for i, (vx, vy) in enumerate(self.vectors):
            if (used_mask >> i) & 1:
                continue

            nx = x + vx
            ny = y + vy

            if abs(nx) > self.b or abs(ny) > self.b:
                continue

            new_dist = self.manhattan(nx, ny)
            gain = old_dist - new_dist

            if gain > best_gain:
                best_gain = gain
                best_i = i

        return best_i, best_gain

    def solve(self):
        x = 0
        y = 0
        used = 0
        turn = 0

        score = 0

        for _ in range(self.n):
            i, gain = self.best_move(x, y, used)

            if i == -1:
                break

            vx, vy = self.vectors[i]

            x += vx
            y += vy
            used |= (1 << i)

            if turn == 0:
                score += gain
            else:
                score -= gain

            if (x, y) == self.target:
                return "PLAYER 0" if turn == 0 else "PLAYER 1"

            if abs(x) > self.b or abs(y) > self.b:
                return "PLAYER 1" if turn == 0 else "PLAYER 0"

            turn ^= 1

        if score > 0:
            return "PLAYER 0"
        if score < 0:
            return "PLAYER 1"
        return "TIE"

In [35]:
from tqdm import tqdm
test_df = pd.read_csv('test_data.csv')

solutions = []
for idx, row in tqdm(test_df.iterrows()):
    game = Game(row)
    solutions.append(game.solve())

50it [00:00, 3694.90it/s]


In [17]:
submission = pd.read_csv('sample_output.csv')

submission['answer'] = solutions

submission.to_csv('submission.csv', index=False)
submission['answer'].value_counts()

answer
TIE    50
Name: count, dtype: int64